# 🚀 Tải, Tiền Xử Lý Dataset MipNerf360 và Upload Lên Kaggle Từ Google Colab

Vì Kaggle giới hạn dung lượng ổ đĩa làm việc `/kaggle/working` là **20GB**, việc vừa tải file zip dataset (17GB) vừa giải nén nó ra sẽ gây ra lỗi hết bộ nhớ (`No space left on device`).

**Giải pháp:** Chạy notebook này trên **Google Colab** (ổ đĩa khả dụng hơn **100GB**), sau đó sử dụng **Kaggle API** để đẩy thẳng bộ dataset đã được giải nén và làm phẳng lên tài khoản Kaggle của bạn với tên **MipNerf360-dataset**.

--- 
### 🔑 Cách tạo và lấy Token Kaggle API mới (Sửa lỗi 401 Unauthorized):
1. Truy cập [Kaggle](https://www.kaggle.com/) và đăng nhập vào tài khoản của bạn.
2. Nhấp vào ảnh đại diện ở góc trên cùng bên phải -> Chọn **Settings**.
3. Cuộn xuống phần **API** -> Nhấn nút **Create New Token**.
4. Trình duyệt sẽ tải về một file tên là `kaggle.json` (chứa `username` và `key` của bạn).
5. Mở file `kaggle.json` đó ra và copy các thông số dán vào **Form cấu hình** ở cell bên dưới.

In [ ]:
#@title ── Cấu Hình Thông Tin Tài Khoản Kaggle & HuggingFace ── { display-mode: "form" }
import os

#@markdown **HuggingFace settings:**
HF_TOKEN = "YOUR_HF_TOKEN" #@param {type:"string"}
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

#@markdown **Kaggle Settings:**
#@markdown *Hãy thay thế username và key bên dưới bằng thông tin trong file `kaggle.json` mới của bạn*
KAGGLE_USERNAME = "nctuan" #@param {type:"string"}
KAGGLE_KEY = "YOUR_KAGGLE_KEY_OR_TOKEN" #@param {type:"string"}

print("✅ Cấu hình hoàn tất!")

In [ ]:
# ── Cài Đặt Kaggle API & Thiết Lập Token Truy Cập ───────────────────────────────
import json
import os
from pathlib import Path

# Dọn dẹp thư viện cũ và cài đặt Kaggle CLI bản mới nhất (để hỗ trợ token mới KGAT_)
!pip uninstall -y -q kaggle kagglesdk kagglehub
!pip install -q kaggle kagglehub --upgrade

# Cấu hình thư mục .kaggle chứa credentials
home_dir = Path.home()
kaggle_dir = home_dir / '.kaggle'
kaggle_dir.mkdir(parents=True, exist_ok=True)

# 1. Phương pháp Legacy (kaggle.json)
config_path = kaggle_dir / 'kaggle.json'
with open(config_path, 'w') as f:
    json.dump({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}, f)
config_path.chmod(0o600)

# 2. Phương pháp Modern (access_token)
token_path = kaggle_dir / 'access_token'
with open(token_path, 'w') as f:
    f.write(KAGGLE_KEY)
token_path.chmod(0o600)

# 3. Thiết lập biến môi trường cho cả 2 phương pháp
os.environ['KAGGLE_USERNAME'] = KAGGLE_USERNAME
os.environ['KAGGLE_KEY'] = KAGGLE_KEY
os.environ['KAGGLE_API_TOKEN'] = KAGGLE_KEY

print(f"✅ Đã thiết lập thông tin Kaggle API tại: {kaggle_dir}")

In [ ]:
# ── Tải Dataset 'testing.zip' Từ HuggingFace Hub Về Colab ───────────────────────
import os
from huggingface_hub import hf_hub_download

zip_dest = "/content/testing.zip"

if not os.path.exists(zip_dest):
    print("⚡ Đang tải testing.zip từ HuggingFace (DiBiay/thesis_dataset) ...")
    try:
        hf_hub_download(
            repo_id="DiBiay/thesis_dataset",
            filename="testing.zip",
            repo_type="dataset",
            token=HF_TOKEN,
            local_dir="/content",
            local_dir_use_symlinks=False
        )
        print(f"✅ Tải hoàn tất! Tệp tin được lưu tại: {zip_dest}")
    except Exception as e:
        print(f"❌ Lỗi khi tải dataset: {e}")
else:
    print(f"✅ Dataset 'testing.zip' đã tồn tại ở: {zip_dest}")

In [ ]:
# ── Giải Nén, Làm Phẳng Cấu Trúc & Dọn Dẹp File Zip ─────────────────────────────
import os
import zipfile
import shutil

zip_dest = "/content/testing.zip"
output_dataset_dir = "/content/MipNerf360-dataset"

print(f"📦 Đang giải nén {zip_dest} vào {output_dataset_dir} ...")
with zipfile.ZipFile(zip_dest, 'r') as zip_ref:
    zip_ref.extractall(output_dataset_dir)
print("✅ Giải nén thành công!")

# ── Làm phẳng cấu trúc thư mục (Flatten scene folders) ───────────────────────────
print("\n🔍 Đang chuẩn hóa cấu trúc scene...")
scenes_moved = 0
scene_paths_to_move = []

# Quét tìm các thư mục chứa thư mục con 'images_8', 'images_4' hoặc 'images'
for root, dirs, files in os.walk(output_dataset_dir):
    if any(d in ["images_8", "images_4", "images"] for d in dirs):
        scene_paths_to_move.append(root)

scene_paths_to_move = list(set(scene_paths_to_move))

for scene_path in scene_paths_to_move:
    scene_name = os.path.basename(scene_path)
    final_scene_dest = os.path.join(output_dataset_dir, scene_name)
    
    if os.path.abspath(scene_path) != os.path.abspath(final_scene_dest):
        print(f"  -> Di chuyển '{scene_name}' từ {scene_path} về {final_scene_dest}")
        if os.path.exists(final_scene_dest):
            shutil.rmtree(final_scene_dest)
        shutil.move(scene_path, final_scene_dest)
        scenes_moved += 1

# Xóa các thư mục cha bị rỗng sau khi chuyển
for root, dirs, files in os.walk(output_dataset_dir, topdown=False):
    for d in dirs:
        dir_to_check = os.path.join(root, d)
        if os.path.exists(dir_to_check) and not os.listdir(dir_to_check):
            os.rmdir(dir_to_check)

print(f"✅ Chuẩn hóa cấu trúc hoàn tất. Đã di chuyển {scenes_moved} scenes.")

# ── Xóa file zip thô để tiết kiệm bộ nhớ ───────────────────────────────────────
if os.path.exists(zip_dest):
    print(f"\n🧹 Đang xóa file zip thô trên Colab: {zip_dest}...")
    os.remove(zip_dest)
    print("✅ Xóa file zip thành công để tối ưu ổ đĩa!")

In [ ]:
# ── Tạo File Metadata & Upload Bộ Dataset Lên Kaggle ───────────────────────────
import json
import subprocess
import os

output_dataset_dir = "/content/MipNerf360-dataset"
metadata_path = os.path.join(output_dataset_dir, "dataset-metadata.json")

# 1. Viết thông tin mô tả metadata cho Kaggle Dataset
metadata = {
    "title": "MipNerf360-dataset",
    "id": f"{KAGGLE_USERNAME}/mipnerf360-dataset",
    "licenses": [
        {
            "name": "CC0-1.0"
        }
    ]
}

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4)

print("✅ Đã tạo file dataset-metadata.json!")
print(json.dumps(metadata, indent=4))

# 2. Tiến hành upload bằng Kaggle CLI
print("\n🚀 Đang upload bộ dataset lên Kaggle Datasets (Vui lòng đợi vài phút)...")

# Thử khởi tạo dataset mới
result = subprocess.run(["kaggle", "datasets", "create", "-p", output_dataset_dir, "-r", "zip"], capture_output=True, text=True)

# Nếu dataset đã tồn tại, tiến hành đẩy một version mới thay thế
if "already exists" in result.stderr or "already exists" in result.stdout:
    print("\n🔄 Bộ dataset đã tồn tại trên tài khoản Kaggle của bạn. Tiến hành tải lên phiên bản mới...")
    result = subprocess.run([
        "kaggle", "datasets", "version", 
        "-p", output_dataset_dir, 
        "-m", "Được tải lên tự động từ Google Colab"
    ], capture_output=True, text=True)

print("\n--- KẾT QUẢ UPLOAD KAGGLE ---")
print("STDOUT:")
print(result.stdout)
if result.stderr:
    print("STDERR:")
    print(result.stderr)

if result.returncode == 0:
    print(f"\n🎉 Chúc mừng! Bạn đã tải dataset lên Kaggle thành công dưới tên: nctuan/mipnerf360-dataset")
    print("👉 Bây giờ bạn chỉ cần add dataset này vào notebook huấn luyện trên Kaggle để tiếp tục chạy!")
else:
    print("\n❌ Có lỗi xảy ra trong quá trình upload. Vui lòng kiểm tra log lỗi ở trên.")